Groundwater | Flow Modeling Track

# Step 8: Model Application — Using the Model to Answer Questions

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → **8-Apply** → 9-Document → 10-Communicate

**Story so far:** You have a calibrated MODFLOW 6 model of the Limmat Valley (Step 5). Cross-validation (Step 6) shows reasonable predictive power near the observation cluster, and FOSM sensitivity analysis (Step 7) highlights that predictions in the data-sparse western domain carry the highest uncertainty. Now it's time to **use the model** — the whole reason we built it.

| **Core content** | ~35 minutes |
|:--|:--|
| **Optional: Drought stress test** | +15 minutes |

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Design** a model scenario with a clear question, controlled changes, and appropriate outputs
2. **Implement** and run a new-pumping-well scenario using the calibrated model
3. **Compute** and interpret drawdown maps and water balance changes
4. **Delineate** well capture zones using MODFLOW 6 particle tracking (PRT)
5. **Relate** capture zones to Swiss groundwater protection zone regulations (S1, S2, S3)
6. **Compare** numerical capture zones with the analytical solution for a confined aquifer
7. *(optional)* Evaluate system response to reduced recharge (climate stress testing)

## Prerequisites

- **Completed [5_calibration.ipynb](5_calibration.ipynb)** — the calibration workspace must exist

In [ ]:
# Setup
import sys
import os
import shutil
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point

from IPython.display import display, HTML

# Add support modules to path
sys.path.append(os.path.abspath('../../_SUPPORT/src'))
sys.path.append(os.path.abspath('../../_SUPPORT/src/scripts'))
sys.path.append(os.path.abspath('../../_SUPPORT/src/scripts/scripts_exercises'))

from data_utils import download_named_file, get_default_data_folder
from model_io_utils import load_base_simulation, format_budget_summary
from boundary_utils import assign_well_cells
from plot_utils import quick_model_plot

# Checkpoint utilities
from shared_functions import check_task_with_solution, create_multiple_choice

import calibration_utils as cu
from setup_pest_calibration import get_calibrated_k_field, load_pest_results

import flopy

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 7)
plt.rcParams['font.size'] = 12

In [ ]:
# --- 1. Define paths and create fresh application workspace ---
DATA_DIR = get_default_data_folder()
sim_name = 'limmat_valley'
nb4_workspace = os.path.join(DATA_DIR, 'notebook4_model')
calib_workspace = os.path.join(DATA_DIR, 'calibration')
app_workspace = os.path.join(DATA_DIR, 'application')

# Check calibration workspace exists (produced by NB5)
if not os.path.exists(os.path.join(calib_workspace, 'mfsim.nam')):
    raise FileNotFoundError(
        f"Calibration workspace not found at {calib_workspace}\n"
        "Please run Notebook 5 first to generate the calibrated model."
    )

# Always start fresh
if os.path.exists(app_workspace):
    shutil.rmtree(app_workspace)
shutil.copytree(calib_workspace, app_workspace)
workspace = app_workspace
print(f"Application workspace: {workspace}")

# --- 2. Load simulation ---
sim = load_base_simulation(workspace)
gwf = sim.get_model(sim_name)
modelgrid = gwf.modelgrid
modelgrid.set_coord_info(crs="EPSG:2056")

disv = gwf.get_package('DISV')
idomain = disv.idomain.array
top = disv.top.array.flatten()
botm = disv.botm.array.flatten()

# Load boundary and river GIS data
boundary_path = download_named_file(name='model_boundary', data_type='gis')
boundary_gdf = gpd.read_file(boundary_path)

river_gis_path = os.path.join(os.path.dirname(boundary_path), 'AV_Gewasser_-OGD.gpkg')
river_all = gpd.read_file(river_gis_path)

# Cell centre coordinates
xc = modelgrid.xcellcenters
yc = modelgrid.ycellcenters

print(f"Grid type: {modelgrid.grid_type}")
print(f"Active cells: {(idomain.flatten() > 0).sum()}")

# --- 3. Apply calibrated K field ---
pest_ws = os.path.join(calib_workspace, 'pest_setup')
k_field_cal, pp_df = get_calibrated_k_field(
    pest_ws, modelgrid,
    variogram_range=6000.0, anisotropy_angle=-45.0, anisotropy_scaling=3.0,
)
gwf.npf.k.set_data(k_field_cal)

# Apply Sihl leakance multiplier from calibration
par_df, _ = load_pest_results(pest_ws)

# Identify Sihl river cells
river_gdf = river_all[
    river_all['GEWAESSERNAME'].isin(['Limmat', 'Sihl'])
    & river_all.intersects(boundary_gdf.geometry.iloc[0])
]

riv = gwf.get_package('RIV')
riv_data = riv.stress_period_data.get_data(0)
sihl_poly = river_gdf[river_gdf['GEWAESSERNAME'] == 'Sihl'].union_all()
sihl_cell_ids = []
for rec in riv_data:
    cid = rec['cellid']
    cell_idx = cid[-1] if isinstance(cid, tuple) else cid
    if sihl_poly.contains(Point(xc[cell_idx], yc[cell_idx])):
        sihl_cell_ids.append(cell_idx)

if 'sihl_leakance_mult' in par_df.index:
    mult_log = par_df.loc['sihl_leakance_mult', 'optimised']
    if not np.isnan(mult_log):
        sihl_mult = 10.0 ** mult_log
        spd = riv.stress_period_data.get_data(0)
        sihl_set = set(sihl_cell_ids)
        for i in range(len(spd)):
            cid = spd[i]['cellid']
            cell_idx = cid[-1] if isinstance(cid, tuple) else cid
            if cell_idx in sihl_set:
                spd[i]['cond'] *= sihl_mult
        riv.stress_period_data.set_data(spd, key=0)
        print(f"Sihl leakance multiplier applied: {sihl_mult:.1f}×")

# --- 4. Run baseline (with specific discharge for later use) ---
gwf.npf.save_specific_discharge = True
gwf.npf.save_flows = True
gwf.npf.save_saturation = True
sim.write_simulation()
success, _ = sim.run_simulation(silent=True)
if not success:
    raise RuntimeError("Baseline model run failed.")

head_baseline = gwf.output.head().get_data()
heads_bl = head_baseline.flatten() if head_baseline.ndim > 1 else head_baseline
print(f"Baseline head range: {heads_bl[idomain.flatten() > 0].min():.2f}"
      f" – {heads_bl[idomain.flatten() > 0].max():.2f} m a.s.l.")

# Save baseline specific discharge (undisturbed flow field — used for analytical overlay)
spdis_baseline = gwf.output.budget().get_data(text='DATA-SPDIS')[0]

# Baseline water balance
budget_baseline = format_budget_summary(gwf, sim)

---

## 1 — Before You Run: Scenario Design Principles

A model is a tool for answering questions. Before running any scenario, work through this checklist:

### Five Questions Before Running Any Scenario

| # | Question | Purpose |
|---|----------|---------|
| 1 | **What is the question?** | Define the prediction target (e.g., "maximum drawdown at location X") |
| 2 | **Can the model answer it?** | Match model physics/resolution to the question |
| 3 | **What are the controlled changes?** | Only modify what the scenario requires |
| 4 | **What is the baseline for comparison?** | Scenarios are meaningless without a reference |
| 5 | **How uncertain is the answer?** | Use NB7 insights to qualify results |

### Our Model's Capability Assessment

| Characteristic | Can do | Cannot do |
|----------------|--------|-----------|
| **Steady-state** | Long-term equilibrium drawdown | Response times, recovery |
| **Single layer** | Depth-averaged flow | Vertical gradients, multi-aquifer systems |
| **Calibrated K (pilot points)** | Spatially varying conductivity | Sub-grid heterogeneity |
| **River boundary (RIV)** | Head-dependent exchange | Overbank flooding, channel morphology |
| **No transport** | Flow paths, travel times (via PRT) | Contaminant concentrations, dispersion |

In [ ]:
# Checkpoint 1: Can our steady-state model predict transient drawdown?
create_multiple_choice('task08_checkpoint_1')

---

## 2 — New Pumping Well Impact

**Scenario:** A water utility proposes a new pumping well in the western part of the Limmat Valley — the data-sparse area identified in NB7 as having the highest parameter uncertainty. We'll assess the steady-state impact on heads and water balance.

**The five questions applied:**
1. *Question:* What is the maximum drawdown and where does the water come from?
2. *Can the model answer it?* Yes — steady-state drawdown and water balance are appropriate uses.
3. *Controlled changes:* Add one WEL entry; keep everything else identical to the baseline.
4. *Baseline:* The calibrated model we just ran.
5. *Uncertainty:* High in the western domain (NB7) — results are indicative, not precise.

In [ ]:
# --- 2.1 Baseline heads and water balance ---
fig, ax = plt.subplots(figsize=(12, 8))
heads_plot = np.where(idomain.flatten() > 0, heads_bl, np.nan)
quick_model_plot(modelgrid, heads_plot, boundary_gdf=boundary_gdf,
                 cmap='coolwarm', colorbar_label='Head (m a.s.l.)',
                 title='Baseline Head Distribution (calibrated model)', ax=ax)
fig.tight_layout()
plt.show()

print("\nBaseline Water Balance:")
display(budget_baseline)

In [ ]:
# --- 2.2 Define proposed well in the western domain ---
# Pick a location in the western third of the active domain (data-sparse area)
active_mask = idomain.flatten() > 0
active_x = xc[active_mask]
active_y = yc[active_mask]

# Western third of the domain, south of the river (main aquifer)
x_threshold = np.percentile(active_x, 33)
west_mask = active_x < x_threshold

# Use the centroid of the western zone as our well location
well_x = float(np.median(active_x[west_mask]))
well_y = float(np.median(active_y[west_mask]))

# Create a GeoDataFrame for the proposed well
pumping_rate = -500.0  # m³/d (extraction)
well_gdf = gpd.GeoDataFrame(
    {'rate': [pumping_rate], 'name': ['proposed_well']},
    geometry=[Point(well_x, well_y)],
    crs='EPSG:2056',
)

# Find the DISV cell using assign_well_cells
well_spd = assign_well_cells(well_gdf, modelgrid, rate_column='rate')
well_cell_id = well_spd[0][0]  # ((layer, cell2d), rate) → cellid tuple
well_cell_flat = well_cell_id[-1] if isinstance(well_cell_id, tuple) else well_cell_id

print(f"Proposed well location: ({well_x:.0f}, {well_y:.0f})")
print(f"DISV cell ID: {well_cell_flat}")
print(f"Pumping rate: {pumping_rate:.0f} m³/d")
print(f"Local K: {k_field_cal.flatten()[well_cell_flat]:.1f} m/d")
print(f"Aquifer thickness: {(top - botm)[well_cell_flat]:.1f} m")

In [ ]:
# --- 2.3 Add well to WEL package and run scenario ---
wel = gwf.get_package('WEL')
original_wel_spd = wel.stress_period_data.get_data(0).copy()  # save for restoration

# Convert existing recarray to list of tuples and append the new well
existing_wells = [(rec['cellid'], rec['q']) for rec in original_wel_spd]
new_spd = existing_wells + [((0, well_cell_flat), pumping_rate)]
wel.stress_period_data.set_data(new_spd, key=0)

# Run scenario
sim.write_simulation()
success_pump, _ = sim.run_simulation(silent=True)
if not success_pump:
    raise RuntimeError("Pumping scenario failed.")

head_pumping = gwf.output.head().get_data()
heads_pump = head_pumping.flatten() if head_pumping.ndim > 1 else head_pumping
print("Pumping scenario completed successfully.")

In [ ]:
# --- 2.4 Compute and plot drawdown ---
drawdown = heads_bl - heads_pump  # positive = head declined
dd_active = drawdown[active_mask]
dd_plot = np.where(active_mask, drawdown, np.nan)

# Maximum drawdown at the well
dd_at_well = drawdown[well_cell_flat]
print(f"Maximum drawdown at well cell: {dd_at_well:.3f} m")
print(f"Drawdown range: {dd_active.min():.4f} – {dd_active.max():.4f} m")

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
vmax = max(abs(dd_active.min()), abs(dd_active.max()))
quick_model_plot(modelgrid, dd_plot, boundary_gdf=boundary_gdf,
                 cmap='RdBu_r', colorbar_label='Drawdown (m)',
                 title=f'Drawdown from New Well (Q = {pumping_rate:.0f} m³/d)',
                 vmin=-vmax, vmax=vmax, ax=ax)

# Mark the well
ax.scatter(well_x, well_y, s=85, c='black', marker='v',
           edgecolors='white', linewidth=1.5, zorder=6, label='New well')
ax.legend(loc='lower left')
fig.tight_layout()
plt.show()

In [ ]:
# Checkpoint 2: Maximum drawdown at the well cell (m)
check_task_with_solution('task08_checkpoint_2')

In [ ]:
# --- 2.5 Water balance comparison ---
budget_pumping = format_budget_summary(gwf, sim)

# Compare key components
print("\n" + "=" * 60)
print("Water Balance Comparison: Baseline vs Pumping")
print("=" * 60)

for component in budget_baseline.index:
    comp_str = component.decode() if isinstance(component, bytes) else str(component)
    if comp_str.strip() in ('TOTAL', 'DISCREPANCY', 'PERCENT ERROR'):
        continue
    if component not in budget_pumping.index:
        continue
    bl_in = budget_baseline.loc[component, 'Inflow (m3/d)']
    bl_out = budget_baseline.loc[component, 'Outflow (m3/d)']
    pm_in = budget_pumping.loc[component, 'Inflow (m3/d)']
    pm_out = budget_pumping.loc[component, 'Outflow (m3/d)']
    if isinstance(bl_in, (int, float)) and isinstance(pm_in, (int, float)):
        d_in = pm_in - bl_in
        d_out = pm_out - bl_out
        if abs(d_in) > 1 or abs(d_out) > 1:
            print(f"  {comp_str:20s}  ΔInflow: {d_in:+8.1f}  ΔOutflow: {d_out:+8.1f} m³/d")

print(f"\n  Pumping rate:       {pumping_rate:+.0f} m³/d")
print(f"  The budget must close: extra outflow ≈ extra inflow from boundaries.")

In [ ]:
# Checkpoint 3: Where does the pumped water come from?
create_multiple_choice('task08_checkpoint_3')

### Interpretation

**What we learned:**
- The drawdown cone is local — it extends a few hundred metres from the well before being absorbed by the river boundary.
- The river responds by increasing leakage into the aquifer, supplying the pumped water.
- The drawdown magnitude depends on the local K (from pilot-point calibration), which has **high uncertainty** in the western domain (see NB7).

**Caveats:**
- This is **equilibrium** drawdown — the actual drawdown would develop over days to weeks.
- The model is **not validated** near the proposed well location.
- Results should be treated as **indicative**, not as design values.

### Exercise: Linearity Check

The steady-state groundwater flow equation $\nabla \cdot (T \nabla h) + W = 0$ is **linear in head**. This means doubling the pumping rate should approximately double the drawdown. Let's verify:

> **Note:** The ratio may deviate slightly from 2.0 because the RIV boundary is **head-dependent** — as heads drop, the river leaks more water into the aquifer. This introduces mild nonlinearity into the full system, even though the governing PDE is linear. In practice, the ratio is close enough to 2.0 that the linear approximation holds well for scenario planning.

In [ ]:
# --- 2.7 Double the pumping rate and compare ---
double_rate = 2 * pumping_rate  # -1000 m³/d

new_spd_double = existing_wells + [((0, well_cell_flat), double_rate)]
wel.stress_period_data.set_data(new_spd_double, key=0)

sim.write_simulation()
success_double, _ = sim.run_simulation(silent=True)

if success_double:
    head_double = gwf.output.head().get_data()
    heads_dbl = head_double.flatten() if head_double.ndim > 1 else head_double
    dd_double = heads_bl - heads_dbl
    dd_double_at_well = dd_double[well_cell_flat]

    print(f"Drawdown at Q = {pumping_rate:.0f} m³/d:   {dd_at_well:.4f} m")
    print(f"Drawdown at Q = {double_rate:.0f} m³/d:  {dd_double_at_well:.4f} m")
    print(f"Ratio: {dd_double_at_well / dd_at_well:.3f}  (expected: 2.0 for linear system)")
else:
    print("Double-pumping scenario failed.")

# Restore original WEL data
wel.stress_period_data.set_data(existing_wells, key=0)
print("\nWEL package restored to baseline.")

In [ ]:
# Checkpoint 4: Why does drawdown scale linearly?
create_multiple_choice('task08_checkpoint_4')

---

## 3 — Capture Zone and Swiss Groundwater Protection Zones

Now we move to the central application: **where does the water pumped by the well come from?** This defines the well's **capture zone** — the area of the aquifer that contributes water to the well. In Switzerland, capture zones are directly linked to regulatory protection zones.

### Swiss Groundwater Protection Zones (Grundwasserschutzzonen, GSchV Art. 29)

| Zone | Name | Definition | Key Restrictions | How We Delineate |
|------|------|-----------|-----------------|------------------|
| **S1** | Fassungsbereich | Immediate well surrounds | No construction, no access | Fixed distance (~10 m) |
| **S2** | Engere Schutzzone | **10-day groundwater travel time** isochrone | No hazardous storage, agricultural restrictions | **Forward particle tracking** |
| **S3** | Weitere Schutzzone | Full capture zone (or defined portion) | Land-use restrictions | **Forward particle tracking** |

**S2 is the most important zone to compute** — it determines where contamination could reach the well within 10 days, too fast for emergency response.

### Grid Refinement for Protection Zone Resolution

The S2 zone typically extends only **20–30 m** from the well — comparable to our base Voronoi cell size of 50 m. With one particle per cell, the coarse grid cannot resolve this critical zone.

We therefore **locally refine the grid** around the well to ~10 m cells within 200 m, using the `refine_grid_locally()` utility from `disv_grid_utils`. Triangle handles the transition from fine to coarse cells automatically.

This requires rebuilding the GWF model on the refined grid before running PRT. You will follow the same workflow for your case study.

### Implementation: MODFLOW 6 Particle Tracking (PRT)

We use the **PRT** (Particle Tracking) model built into MODFLOW 6. PRT reads the flow field from our GWF simulation and tracks particles through the velocity field — cell by cell, face by face — using the actual heterogeneous K field and boundary conditions.

**Forward tracking for capture zones:** We release one particle at the centre of every active cell and track forward. Particles that terminate at the pumping well define its capture zone. The travel time from each particle's release point to the well gives us the isochrone data for protection zones.

| PRT Component | Purpose |
|---------------|---------|
| **DISV** | Same Voronoi grid as the GWF model |
| **MIP** | Porosity and tracking zones (well = zone 1, river = zone 2) |
| **PRP** | Particle release points (one per active cell) |
| **FMI** | Links to GWF head and budget output files |
| **EMS** | Explicit (non-iterative) solution for particle tracking |

In [ ]:
# --- 3.1 Build locally-refined grid and GWF model ---
from scipy.interpolate import NearestNDInterpolator
from shapely.geometry import Polygon as ShaPoly
from shapely.prepared import prep as shapely_prep
import disv_grid_utils as dgu

# --- Create refined Voronoi grid (10 m cells within 200 m of well) ---
print("Building refined Voronoi grid (~10 m cells near well)...")
base_grid = dgu.create_disv_from_boundary(
    boundary_gdf=boundary_gdf, cell_size=50, crs='EPSG:2056',
)
refine_circle = Point(well_x, well_y).buffer(200)
refine_gdf = gpd.GeoDataFrame(geometry=[refine_circle], crs='EPSG:2056')

refined = dgu.refine_grid_locally(
    base_grid=base_grid,
    refinement_gdf=refine_gdf,
    refined_cell_size=10,
    boundary_gdf=boundary_gdf,
)

gridprops_ref = refined['disv_gridprops']
ncpl_ref = refined['ncpl']
mg_ref = refined['modelgrid']
xc_ref = mg_ref.xcellcenters
yc_ref = mg_ref.ycellcenters

near_well = np.sqrt((xc_ref - well_x)**2 + (yc_ref - well_y)**2) < 30
print(f"Refined grid: {ncpl_ref} cells (original: {(idomain.flatten() > 0).sum()})")
print(f"  Cells within 30 m of well: {near_well.sum()}")

# --- Interpolate properties from original grid (nearest-neighbour) ---
active_orig = idomain.flatten() > 0
src_pts = list(zip(xc[active_orig], yc[active_orig]))

k_ref = NearestNDInterpolator(src_pts, k_field_cal.flatten()[active_orig])(xc_ref, yc_ref)
top_ref = NearestNDInterpolator(src_pts, top[active_orig])(xc_ref, yc_ref)
botm_ref = NearestNDInterpolator(src_pts, botm[active_orig])(xc_ref, yc_ref)
strt_ref = NearestNDInterpolator(src_pts, heads_pump[active_orig])(xc_ref, yc_ref)

# Clip starting heads: must be above bottom (interpolation can produce inconsistencies)
strt_ref = np.maximum(strt_ref, botm_ref + 0.01)

rch_orig = gwf.get_package('RCHA').recharge.get_data()
rch_vals = (list(rch_orig.values())[0] if isinstance(rch_orig, dict) else rch_orig)
rch_ref = NearestNDInterpolator(src_pts, rch_vals.flatten()[active_orig])(xc_ref, yc_ref)

# --- Build GWF simulation on refined grid ---
ref_ws = os.path.join(workspace, 'refined')
if os.path.exists(ref_ws):
    shutil.rmtree(ref_ws)
os.makedirs(ref_ws)

ref_sim_name = 'limmat_refined'
ref_sim = flopy.mf6.MFSimulation(
    sim_name=ref_sim_name, exe_name='mf6', sim_ws=ref_ws,
)
flopy.mf6.ModflowTdis(ref_sim, time_units='DAYS', nper=1,
                        perioddata=[(1.0, 1, 1)])
flopy.mf6.ModflowIms(ref_sim, complexity='MODERATE',
                       outer_maximum=200, inner_maximum=100,
                       outer_dvclose=1e-6, inner_dvclose=1e-9)

ref_gwf = flopy.mf6.ModflowGwf(ref_sim, modelname=ref_sim_name, save_flows=True)

flopy.mf6.ModflowGwfdisv(
    ref_gwf, nlay=1, ncpl=ncpl_ref, nvert=gridprops_ref['nvert'],
    top=top_ref, botm=[botm_ref],
    vertices=gridprops_ref['vertices'], cell2d=gridprops_ref['cell2d'],
)
flopy.mf6.ModflowGwfnpf(ref_gwf, k=k_ref, save_flows=True,
                          save_specific_discharge=True, save_saturation=True)
flopy.mf6.ModflowGwfic(ref_gwf, strt=strt_ref)
flopy.mf6.ModflowGwfrcha(ref_gwf, recharge=rch_ref)

# CHD — transfer constant-head boundary from original grid
chd_orig = gwf.get_package('CHD')
chd_data = chd_orig.stress_period_data.get_data(0)
chd_xy = np.array([
    (xc[r['cellid'][-1] if isinstance(r['cellid'], tuple) else r['cellid']],
     yc[r['cellid'][-1] if isinstance(r['cellid'], tuple) else r['cellid']])
    for r in chd_data
])
head_nn = NearestNDInterpolator(chd_xy, [r['head'] for r in chd_data])

chd_spd_ref = []
for r in chd_data:
    cid = r['cellid'][-1] if isinstance(r['cellid'], tuple) else r['cellid']
    ox, oy = xc[cid], yc[cid]
    dists = np.sqrt((xc_ref - ox)**2 + (yc_ref - oy)**2)
    near = np.where(dists < 30)[0]
    for j in near:
        chd_spd_ref.append(((0, int(j)), float(head_nn(xc_ref[j], yc_ref[j]))))

seen = set()
chd_spd_dedup = []
for rec in chd_spd_ref:
    if rec[0] not in seen:
        seen.add(rec[0])
        chd_spd_dedup.append(rec)
flopy.mf6.ModflowGwfchd(ref_gwf, stress_period_data=chd_spd_dedup)
print(f"CHD cells: {len(chd_spd_dedup)}")

# RIV — identify river cells in refined grid
riv_orig = gwf.get_package('RIV').stress_period_data.get_data(0)
riv_xy = np.array([
    (xc[r['cellid'][-1] if isinstance(r['cellid'], tuple) else r['cellid']],
     yc[r['cellid'][-1] if isinstance(r['cellid'], tuple) else r['cellid']])
    for r in riv_orig
])
stage_nn = NearestNDInterpolator(riv_xy, [r['stage'] for r in riv_orig])
rbot_nn = NearestNDInterpolator(riv_xy, [r['rbot'] for r in riv_orig])
riv_cond = np.array([r['cond'] for r in riv_orig])

orig_riv_areas = np.array([
    ShaPoly(modelgrid.get_cell_vertices(
        r['cellid'][-1] if isinstance(r['cellid'], tuple) else r['cellid']
    )).area for r in riv_orig
])

river_union = river_gdf[
    river_gdf.intersects(boundary_gdf.geometry.iloc[0])
].geometry.union_all()
river_prep = shapely_prep(river_union)

riv_spd_ref = []
for i in range(ncpl_ref):
    if river_prep.contains(Point(xc_ref[i], yc_ref[i])):
        dists = (riv_xy[:, 0] - xc_ref[i])**2 + (riv_xy[:, 1] - yc_ref[i])**2
        j = np.argmin(dists)
        area_new = ShaPoly(mg_ref.get_cell_vertices(i)).area
        cond_scaled = riv_cond[j] * area_new / orig_riv_areas[j]
        riv_spd_ref.append((
            (0, i), float(stage_nn(xc_ref[i], yc_ref[i])),
            float(cond_scaled), float(rbot_nn(xc_ref[i], yc_ref[i])),
        ))

flopy.mf6.ModflowGwfriv(ref_gwf, stress_period_data=riv_spd_ref)
print(f"RIV cells: {len(riv_spd_ref)}")

# WEL
well_cell_ref = int(np.argmin((xc_ref - well_x)**2 + (yc_ref - well_y)**2))
flopy.mf6.ModflowGwfwel(ref_gwf, stress_period_data=[((0, well_cell_ref), pumping_rate)])
print(f"Well cell: {well_cell_ref}")

# OC
flopy.mf6.ModflowGwfoc(
    ref_gwf,
    head_filerecord=[f'{ref_sim_name}.hds'],
    budget_filerecord=[f'{ref_sim_name}.cbc'],
    saverecord=[('HEAD', 'ALL'), ('BUDGET', 'ALL')],
)

# --- Run refined model ---
print("Running refined GWF model...")
ref_sim.write_simulation()
success_ref, _ = ref_sim.run_simulation(silent=True)
if not success_ref:
    raise RuntimeError("Refined GWF model failed — check listing file.")

heads_ref = ref_gwf.output.head().get_data().flatten()
ref_gwf.modelgrid.set_coord_info(crs="EPSG:2056")
print(f"Refined model completed. Head range: "
      f"{heads_ref.min():.2f} – {heads_ref.max():.2f} m")

# Drawdown and heads on refined grid (for plotting in later cells)
heads_bl_ref = NearestNDInterpolator(src_pts, heads_bl[active_orig])(xc_ref, yc_ref)
dd_ref_plot = heads_bl_ref - heads_ref
heads_ref_plot = heads_ref
vmax_ref = np.max(np.abs(dd_ref_plot))

In [ ]:
# --- 3.2 Set up MODFLOW 6 Particle Tracking (PRT) on refined grid ---
from shapely.geometry import Polygon as ShapelyPolygon

# Porosity for alluvial gravels
porosity = 0.15

# --- Create PRT simulation ---
prt_ws = os.path.join(ref_ws, 'prt')
if os.path.exists(prt_ws):
    shutil.rmtree(prt_ws)
os.makedirs(prt_ws)
prt_name = 'limmat_prt'

prt_sim = flopy.mf6.MFSimulation(
    sim_name=prt_name, exe_name='mf6', sim_ws=prt_ws,
)
flopy.mf6.ModflowTdis(
    prt_sim, time_units='DAYS', nper=1, perioddata=[(365.0, 1, 1)],
)

# PRT model
prt = flopy.mf6.ModflowPrt(prt_sim, modelname=prt_name)

# DISV — copy grid from refined GWF model
disv_ref = ref_gwf.get_package('DISV')
flopy.mf6.ModflowPrtdisv(
    prt,
    nlay=disv_ref.nlay.data,
    ncpl=disv_ref.ncpl.data,
    nvert=disv_ref.nvert.data,
    top=disv_ref.top.array,
    botm=disv_ref.botm.array,
    vertices=disv_ref.vertices.array,
    cell2d=disv_ref.cell2d.array,
)

# MIP — porosity and tracking zones
izone_ref = np.zeros((1, ncpl_ref), dtype=int)
izone_ref[0, well_cell_ref] = 1  # well = zone 1
riv_ref = ref_gwf.get_package('RIV')
for rec in riv_ref.stress_period_data.get_data(0):
    cid = rec['cellid']
    cell_idx = cid[-1] if isinstance(cid, tuple) else cid
    izone_ref[0, cell_idx] = 2  # river = zone 2

flopy.mf6.ModflowPrtmip(prt, porosity=porosity, izone=izone_ref)

# PRP — release one particle per active cell
# Use polygon representative_point() to guarantee point is inside cell
ref_modelgrid = ref_gwf.modelgrid
active_cells_ref = np.arange(ncpl_ref)
release_x = np.zeros(len(active_cells_ref))
release_y = np.zeros(len(active_cells_ref))
for j, cell in enumerate(active_cells_ref):
    verts = ref_modelgrid.get_cell_vertices(cell)
    poly = ShapelyPolygon(verts)
    rp = poly.representative_point()
    release_x[j] = rp.x
    release_y[j] = rp.y

prt_data = [
    (i, 0, int(cell), float(rx), float(ry), 0.5)
    for i, (cell, rx, ry) in enumerate(zip(active_cells_ref, release_x, release_y))
]

trackcsv_file = f'{prt_name}.trk.csv'

flopy.mf6.ModflowPrtprp(
    prt,
    nreleasepts=len(prt_data),
    packagedata=prt_data,
    local_z=True,
    perioddata={0: ['FIRST']},
    exit_solve_tolerance=1e-5,
    extend_tracking=True,
    stoptraveltime=365.0,
)

# OC — output control
flopy.mf6.ModflowPrtoc(
    prt,
    budget_filerecord=[f'{prt_name}.cbc'],
    track_filerecord=[f'{prt_name}.trk'],
    trackcsv_filerecord=[trackcsv_file],
    saverecord=[('BUDGET', 'ALL')],
)

# FMI — link to refined GWF head and budget output
gwf_hds_ref = os.path.join(ref_ws, f'{ref_sim_name}.hds')
gwf_cbc_ref = os.path.join(ref_ws, f'{ref_sim_name}.cbc')
fmi_pd = [
    ('GWFHEAD',   os.path.relpath(gwf_hds_ref, prt_ws)),
    ('GWFBUDGET', os.path.relpath(gwf_cbc_ref, prt_ws)),
]
flopy.mf6.ModflowPrtfmi(prt, packagedata=fmi_pd)

# EMS — Explicit Model Solution
ems = flopy.mf6.ModflowEms(prt_sim, filename=f'{prt_name}.ems')
prt_sim.register_solution_package(ems, [prt.name])

print(f"PRT model workspace: {prt_ws}")
print(f"  Particles: {len(prt_data)} (one per cell in refined grid)")
print(f"  Porosity: {porosity}")
print(f"  Max tracking time: 365 days")
print(f"  Zones: well = 1, river = 2")

In [ ]:
# --- 3.3 Run PRT and load results ---
prt_sim.write_simulation()
success_prt, buff_prt = prt_sim.run_simulation()
if not success_prt:
    raise RuntimeError("PRT simulation failed — check listing file.")

# Load pathline CSV
prt_df = pd.read_csv(os.path.join(prt_ws, trackcsv_file))

# Transform model-local coordinates to world CRS for plotting
x_world, y_world = ref_modelgrid.get_coords(prt_df['x'].values, prt_df['y'].values)
prt_df['x_world'] = x_world
prt_df['y_world'] = y_world

# Identify particles captured by the well (izone == 1 in their terminal record)
terminal = prt_df.groupby('irpt').last()
captured_ids = set(terminal.loc[terminal['izone'] == 1].index)

# Build pathlines for captured particles
captured_df = prt_df[prt_df['irpt'].isin(captured_ids)].copy()

# Starting positions of captured particles (their release cells = capture zone)
start_records = captured_df.groupby('irpt').first()

print(f"PRT completed: {prt_df['irpt'].nunique()} particles tracked")
print(f"  Captured by well: {len(captured_ids)}")
print(f"  Reached river:    {(terminal['izone'] == 2).sum()}")
print(f"  Other termination: {(~terminal['izone'].isin([1, 2])).sum()}")
if len(captured_ids) > 0:
    travel_times = terminal.loc[terminal['izone'] == 1, 't']
    print(f"\nTravel times to well:")
    print(f"  Min:    {travel_times.min():.1f} days")
    print(f"  Median: {travel_times.median():.1f} days")
    print(f"  Max:    {travel_times.max():.1f} days")

In [ ]:
# --- 3.4 Plot the capture zone (forward pathlines → well) ---
if len(captured_ids) > 0:
    fig, ax = plt.subplots(figsize=(12, 8))

    # Base map: drawdown on refined grid (shows refined cell structure)
    quick_model_plot(mg_ref, dd_ref_plot, boundary_gdf=boundary_gdf,
                     cmap='RdBu_r', colorbar_label='Drawdown (m)',
                     vmin=-vmax_ref, vmax=vmax_ref,
                     title='Capture Zone (PRT forward pathlines, 1 year)',
                     ax=ax)

    # Plot pathlines coloured by travel time
    for pid in captured_ids:
        pl = captured_df[captured_df['irpt'] == pid]
        ax.plot(pl['x_world'].values, pl['y_world'].values,
                color='blue', linewidth=0.8, alpha=0.6)

    # Mark particle start positions (source locations)
    ax.scatter(start_records['x_world'], start_records['y_world'],
               s=15, c='blue', alpha=0.6, zorder=5,
               label='Particle origins (capture zone)')

    # Mark well
    ax.scatter(well_x, well_y, s=85, c='black', marker='v',
               edgecolors='white', linewidth=1.5, zorder=6, label='Pumping well')

    # Zoom to capture zone extent
    all_x = captured_df['x_world'].values
    all_y = captured_df['y_world'].values
    pad = 200
    ax.set_xlim(min(all_x.min(), well_x) - pad, max(all_x.max(), well_x) + pad)
    ax.set_ylim(min(all_y.min(), well_y) - pad, max(all_y.max(), well_y) + pad)

    ax.legend(loc='lower left')
    fig.tight_layout()
    plt.show()
else:
    print("No particles captured by the well — check model setup.")

### Analytical Comparison: Capture Zone in Uniform Flow

For a confined aquifer with uniform flow, the capture zone of a pumping well has a classical analytical solution (Bear, 1979). The **dividing streamline** — the boundary between water captured by the well and water flowing past — satisfies:

$$x = \frac{y}{\tan\!\left(\dfrac{2\pi\, T\, i\, y}{Q}\right)}$$

where $T = Kb$ is transmissivity, $i$ is the regional hydraulic gradient, and $Q$ is the pumping rate. Key properties:

- **Stagnation point** (downstream limit): $x_s = \dfrac{Q}{2\pi\, T\, i}$
- **Maximum half-width** (far upstream): $y_{\max} = \pm\dfrac{Q}{2\, T\, i}$

We'll compute the local $T$ and $i$ from our model and overlay the analytical solution on the PRT capture zone. Deviations show the effect of heterogeneity, boundaries, and non-uniform flow — exactly the reasons we need numerical models.

In [ ]:
# --- 3.5 Analytical capture zone overlay ---
if len(captured_ids) > 0:
    # Local aquifer properties at the well
    k_local = k_field_cal.flatten()[well_cell_flat]
    b_local = float((top - botm)[well_cell_flat])
    T_local = k_local * b_local
    Q = abs(pumping_rate)

    # Regional flow direction from baseline specific discharge (undisturbed field)
    qx_bl = spdis_baseline['qx'][well_cell_flat]
    qy_bl = spdis_baseline['qy'][well_cell_flat]
    q_mag = np.sqrt(qx_bl**2 + qy_bl**2)

    # Hydraulic gradient: q = K * i  →  |i| = |q| / K
    i_local = q_mag / k_local
    # Flow direction = direction of specific discharge vector
    flow_angle = np.arctan2(qy_bl, qx_bl)

    print(f"Local properties at well:")
    print(f"  K = {k_local:.1f} m/d,  b = {b_local:.1f} m,  T = {T_local:.0f} m²/d")
    print(f"  Baseline specific discharge: qx={qx_bl:.5f}, qy={qy_bl:.5f} m/d")
    print(f"  Hydraulic gradient |i| = {i_local:.5f}")
    print(f"  Flow direction: {np.degrees(flow_angle):.1f}° from east")

    # Analytical capture zone (Bear, 1979) — in flow-aligned coordinates
    # +x is the flow direction; stagnation point is downstream (+x)
    x_stag = Q / (2 * np.pi * T_local * i_local)
    y_max = Q / (2 * T_local * i_local)
    print(f"\nAnalytical predictions:")
    print(f"  Stagnation point: {x_stag:.0f} m downstream")
    print(f"  Max capture width: ±{y_max:.0f} m")

    # Compute analytical dividing streamline (in flow-aligned coordinates)
    y_range = np.linspace(-0.95 * y_max, 0.95 * y_max, 500)
    y_range = y_range[y_range != 0]
    arg = 2 * np.pi * T_local * i_local * y_range / Q
    x_analytical = y_range / np.tan(arg)

    # Rotate to model coordinates (x_analytical is along flow direction)
    cos_f = np.cos(flow_angle)
    sin_f = np.sin(flow_angle)
    x_model = well_x + x_analytical * cos_f - y_range * sin_f
    y_model = well_y + x_analytical * sin_f + y_range * cos_f

    # Plot comparison on refined grid
    fig, ax = plt.subplots(figsize=(12, 8))
    quick_model_plot(mg_ref, dd_ref_plot, boundary_gdf=boundary_gdf,
                     cmap='RdBu_r', colorbar_label='Drawdown (m)',
                     vmin=-vmax_ref, vmax=vmax_ref,
                     title='Capture Zone: Numerical (PRT) vs Analytical',
                     ax=ax)

    # Numerical pathlines
    for pid in captured_ids:
        pl = captured_df[captured_df['irpt'] == pid]
        ax.plot(pl['x_world'].values, pl['y_world'].values,
                color='blue', linewidth=0.6, alpha=0.4)

    # Analytical dividing streamline
    ax.plot(x_model, y_model, color='orange', linewidth=2.5, linestyle='--',
            label=f'Analytical (T={T_local:.0f}, i={i_local:.4f})', zorder=5)

    # Well marker
    ax.scatter(well_x, well_y, s=85, c='black', marker='v',
               edgecolors='white', linewidth=1.5, zorder=6, label='Pumping well')

    # Zoom to capture zone extent
    all_x = np.concatenate([captured_df['x_world'].values, x_model])
    all_y = np.concatenate([captured_df['y_world'].values, y_model])
    pad = 200
    ax.set_xlim(min(all_x.min(), well_x) - pad, max(all_x.max(), well_x) + pad)
    ax.set_ylim(min(all_y.min(), well_y) - pad, max(all_y.max(), well_y) + pad)

    ax.legend(loc='lower left', fontsize=9)
    fig.tight_layout()
    plt.show()

    print("\nThe analytical solution assumes uniform flow in a homogeneous confined aquifer.")
    print("Deviations from the numerical pathlines reflect heterogeneous K, non-uniform")
    print("gradients, and proximity to river boundaries — the reasons we need numerical models.")
else:
    print("No pathlines available for comparison.")

In [ ]:
# --- 3.6 Delineate the 10-day isochrone (S2 boundary) ---
from shapely.geometry import MultiPoint as ShapelyMultiPoint
from shapely import concave_hull

if len(captured_ids) > 0:
    # S2: particles that reach the well within 10 days
    travel_to_well = terminal.loc[terminal['izone'] == 1, 't']
    s2_ids = set(travel_to_well[travel_to_well <= 10.0].index)

    print(f"10-day isochrone (S2): {len(s2_ids)} particles reach well within 10 days")

    # Collect all pathline points for S2 particles
    s2_pathlines = captured_df[captured_df['irpt'].isin(s2_ids)]
    s2_pts_x = s2_pathlines['x_world'].values
    s2_pts_y = s2_pathlines['y_world'].values

    if len(s2_ids) > 0:
        s2_dist = np.sqrt((s2_pts_x - well_x)**2 + (s2_pts_y - well_y)**2)
        print(f"  Max extent: {s2_dist.max():.0f} m from well")

        # Convex hull of all S2 pathline points
        s2_cloud = ShapelyMultiPoint(list(zip(s2_pts_x, s2_pts_y)))
        s2_envelope = s2_cloud.convex_hull
    else:
        s2_dist = np.array([])
        s2_envelope = None

    # S3: concave envelope of all captured pathline points
    s3_pts_x = captured_df['x_world'].values
    s3_pts_y = captured_df['y_world'].values
    s3_cloud = ShapelyMultiPoint(list(zip(s3_pts_x, s3_pts_y)))
    s3_envelope = concave_hull(s3_cloud, ratio=0.2)

    # Plot protection zones on refined grid (zoomed to capture zone)
    fig, ax = plt.subplots(figsize=(12, 8))
    quick_model_plot(mg_ref, heads_ref_plot, boundary_gdf=boundary_gdf,
                     cmap='coolwarm', colorbar_label='Head (m a.s.l.)',
                     title='Swiss Groundwater Protection Zones',
                     ax=ax)

    # S3: concave envelope of full capture zone
    if s3_envelope is not None and not s3_envelope.is_empty:
        s3_x_env, s3_y_env = s3_envelope.exterior.xy
        ax.fill(s3_x_env, s3_y_env, facecolor='#2196F3', alpha=0.15,
                edgecolor='#2196F3', linewidth=2.0, zorder=4,
                label='S3: capture zone')

    # S3 pathlines (on top of envelope for detail)
    for pid in captured_ids:
        pl = captured_df[captured_df['irpt'] == pid]
        ax.plot(pl['x_world'].values, pl['y_world'].values,
                color='#2196F3', linewidth=0.5, alpha=0.3)

    # S2: convex envelope of 10-day pathlines
    if s2_envelope is not None and not s2_envelope.is_empty:
        s2_x_env, s2_y_env = s2_envelope.exterior.xy
        ax.fill(s2_x_env, s2_y_env, facecolor='#FF9800', alpha=0.35,
                edgecolor='#FF9800', linewidth=2.0, zorder=5,
                label='S2: 10-day isochrone')

    # S1: 10 m buffer around well (filled for visibility at zoomed scale)
    s1_circle = plt.Circle((well_x, well_y), 10, fill=True,
                            facecolor='red', alpha=0.4,
                            edgecolor='red', linewidth=2.5,
                            label='S1: 10 m radius', zorder=6)
    ax.add_patch(s1_circle)

    # Well marker
    ax.scatter(well_x, well_y, s=85, c='black', marker='v',
               edgecolors='white', linewidth=1.5, zorder=7, label='Pumping well')

    # Zoom to capture zone extent
    all_x = captured_df['x_world'].values
    all_y = captured_df['y_world'].values
    pad = 200
    ax.set_xlim(min(all_x.min(), well_x) - pad, max(all_x.max(), well_x) + pad)
    ax.set_ylim(min(all_y.min(), well_y) - pad, max(all_y.max(), well_y) + pad)

    ax.legend(loc='lower left', fontsize=9)
    fig.tight_layout()
    plt.show()

    print("\nProtection zone summary:")
    print(f"  S1 (Fassungsbereich): 10 m radius — no construction, no activities")
    if s2_envelope is not None and not s2_envelope.is_empty:
        print(f"  S2 (Engere Schutzzone): 10-day travel time — "
              f"area {s2_envelope.area:.0f} m², max extent {s2_dist.max():.0f} m from well")
    else:
        print(f"  S2 (Engere Schutzzone): no particles reached well within 10 days")
    print(f"  S3 (Weitere Schutzzone): full capture zone — "
          f"area {s3_envelope.area:.0f} m², {len(captured_ids)} cells contribute")
else:
    print("No pathlines available.")

In [ ]:
# Checkpoint 5: Farm within the 10-day isochrone — which zone?
create_multiple_choice('task08_checkpoint_5')

### Non-Stationarity of Capture Zones

> **Important:** The capture zone we just computed is based on a **single steady-state flow field** — one snapshot of long-term average conditions.

In reality, capture zones are **non-stationary**:

| Condition | Effect on Capture Zone |
|-----------|----------------------|
| **Summer drought** (low recharge, low river stage) | **Expands** — lower gradients, well draws from larger area |
| **Winter high-flow** (high recharge, high river stage) | **Shrinks** — steeper gradients push water past the well faster |
| **Parameter uncertainty** (different plausible K fields) | **Ensemble of zones** — each K realisation gives a different boundary |

**Regulatory implication:** Swiss guidelines recommend basing protection zones on the **most conservative** (largest) capture zone:
- Use the **dry-season** scenario (lowest gradients → largest zone)
- Or use **Monte Carlo** with K-field realisations from PEST++ IES to produce a probabilistic boundary
- The analytical solution confirms this: $y_{\max} = Q/(2Ti)$ — if gradient $i$ decreases, capture width increases

**Connection to NB7:** The data-sparse western domain has the highest K uncertainty → capture zone uncertainty is largest precisely where we placed the well.

In [ ]:
# Checkpoint 6: Why not base zones on a single steady-state run?
create_multiple_choice('task08_checkpoint_6')

---

## 4 — Optional: Drought Stress Test (~15 min)

**Scenario:** Climate projections suggest 20–30% recharge reduction in coming decades. What happens to aquifer levels and the capture zone?

**The five questions:**
1. *Question:* How much do heads decline, and how does the capture zone change?
2. *Can the model answer it?* Partially — steady-state gives the new equilibrium, not the transient path.
3. *Controlled changes:* Reduce recharge by 30%; keep pumping, river, and boundaries the same.
4. *Baseline:* The pumping-well scenario (so we see the combined effect).
5. *Uncertainty:* Recharge is already uncertain; 30% reduction is a plausible stress test.

In [ ]:
# --- 4.2 Run drought scenario: 30% recharge reduction ---
# Restore pumping well in WEL package
new_spd_pump = existing_wells + [((0, well_cell_flat), pumping_rate)]
wel.stress_period_data.set_data(new_spd_pump, key=0)

# Use run_calibration_trial with rch_multiplier (auto-restores after run)
head_drought = cu.run_calibration_trial(
    sim, sim_name,
    rch_multiplier=0.7,  # 30% reduction
)

if head_drought is not None:
    heads_dr = head_drought.flatten() if head_drought.ndim > 1 else head_drought
    print("Drought scenario completed.")
    print(f"Head range: {heads_dr[active_mask].min():.2f}"
          f" – {heads_dr[active_mask].max():.2f} m a.s.l.")
else:
    print("Drought scenario FAILED.")

In [ ]:
# --- 4.3 Head decline map (drought + pumping vs baseline) ---
if head_drought is not None:
    decline = heads_bl - heads_dr  # positive = head dropped
    decline_plot = np.where(active_mask, decline, np.nan)
    decline_active = decline[active_mask]
    mean_decline = np.nanmean(decline_active)

    print(f"Mean head decline: {mean_decline:.3f} m")
    print(f"Max head decline:  {decline_active.max():.3f} m")
    print(f"Min head decline:  {decline_active.min():.3f} m")

    fig, ax = plt.subplots(figsize=(12, 8))
    quick_model_plot(modelgrid, decline_plot, boundary_gdf=boundary_gdf,
                     cmap='YlOrRd', colorbar_label='Head Decline (m)',
                     title='Head Decline: Drought (−30% recharge) + Pumping Well',
                     ax=ax)
    ax.scatter(well_x, well_y, s=85, c='black', marker='v',
               edgecolors='white', linewidth=1.5, zorder=6, label='New well')
    ax.legend(loc='lower left')
    fig.tight_layout()
    plt.show()

In [ ]:
# Checkpoint 7: Mean head decline (m) across active domain
check_task_with_solution('task08_checkpoint_7')

In [ ]:
# --- 4.4 Water balance under drought ---
# Re-run the drought scenario to get budget (run_calibration_trial restored params)
rch = gwf.get_package('RCHA')
original_rch_data = copy.deepcopy(rch.recharge.get_data())

# Apply 30% reduction
rch_data = rch.recharge.get_data()
if isinstance(rch_data, dict):
    new_rch = {k: v * 0.7 for k, v in rch_data.items()}
else:
    new_rch = rch_data * 0.7
rch.recharge.set_data(new_rch)

sim.write_simulation()
success_dr2, _ = sim.run_simulation(silent=True)

if success_dr2:
    budget_drought = format_budget_summary(gwf, sim)

    print("\n" + "=" * 60)
    print("Water Balance: Baseline vs Drought+Pumping")
    print("=" * 60)
    for component in budget_baseline.index:
        comp_str = component.decode() if isinstance(component, bytes) else str(component)
        if comp_str.strip() in ('TOTAL', 'DISCREPANCY', 'PERCENT ERROR'):
            continue
        if component not in budget_drought.index:
            continue
        bl_net = budget_baseline.loc[component, 'Net (m3/d)']
        dr_net = budget_drought.loc[component, 'Net (m3/d)']
        if isinstance(bl_net, (int, float)) and isinstance(dr_net, (int, float)):
            diff = dr_net - bl_net
            if abs(diff) > 1:
                print(f"  {comp_str:20s}  ΔNet: {diff:+8.1f} m³/d")

# Restore recharge
rch.recharge.set_data(original_rch_data)
print("\nRecharge restored to calibrated values.")

### Stretch Exercise: Capture Zone Under Drought

Re-run the PRT particle tracking under drought conditions. This directly illustrates the non-stationarity discussed above: lower gradients → larger capture zone.

In [ ]:
# --- 4.5 Re-run PRT under drought conditions (on refined grid) ---
if len(captured_ids) > 0:
    # Apply drought recharge to the refined model
    rch_pkg_ref = ref_gwf.get_package('RCHA')
    rch_data_ref = rch_pkg_ref.recharge.get_data()
    original_rch_ref = copy.deepcopy(rch_data_ref)
    if isinstance(rch_data_ref, dict):
        new_rch = {k: v * 0.7 for k, v in rch_data_ref.items()}
    else:
        new_rch = rch_data_ref * 0.7
    rch_pkg_ref.recharge.set_data(new_rch)

    # Run refined GWF with drought conditions
    ref_sim.write_simulation()
    success_dr_mp, _ = ref_sim.run_simulation(silent=True)

    if success_dr_mp:
        # Build drought PRT simulation on refined grid
        prt_ws_dr = os.path.join(ref_ws, 'prt_drought')
        if os.path.exists(prt_ws_dr):
            shutil.rmtree(prt_ws_dr)
        os.makedirs(prt_ws_dr)
        prt_name_dr = 'limmat_prt_dr'

        prt_sim_dr = flopy.mf6.MFSimulation(
            sim_name=prt_name_dr, exe_name='mf6', sim_ws=prt_ws_dr,
        )
        flopy.mf6.ModflowTdis(
            prt_sim_dr, time_units='DAYS', nper=1, perioddata=[(365.0, 1, 1)],
        )
        prt_dr = flopy.mf6.ModflowPrt(prt_sim_dr, modelname=prt_name_dr)

        flopy.mf6.ModflowPrtdisv(
            prt_dr,
            nlay=disv_ref.nlay.data, ncpl=disv_ref.ncpl.data,
            nvert=disv_ref.nvert.data,
            top=disv_ref.top.array, botm=disv_ref.botm.array,
            vertices=disv_ref.vertices.array,
            cell2d=disv_ref.cell2d.array,
        )
        flopy.mf6.ModflowPrtmip(prt_dr, porosity=porosity, izone=izone_ref)

        trackcsv_dr = f'{prt_name_dr}.trk.csv'
        flopy.mf6.ModflowPrtprp(
            prt_dr,
            nreleasepts=len(prt_data), packagedata=prt_data,
            local_z=True, perioddata={0: ['FIRST']},
            exit_solve_tolerance=1e-5, extend_tracking=True,
            stoptraveltime=365.0,
        )
        flopy.mf6.ModflowPrtoc(
            prt_dr,
            budget_filerecord=[f'{prt_name_dr}.cbc'],
            track_filerecord=[f'{prt_name_dr}.trk'],
            trackcsv_filerecord=[trackcsv_dr],
            saverecord=[('BUDGET', 'ALL')],
        )
        fmi_pd_dr = [
            ('GWFHEAD',   os.path.relpath(gwf_hds_ref, prt_ws_dr)),
            ('GWFBUDGET', os.path.relpath(gwf_cbc_ref, prt_ws_dr)),
        ]
        flopy.mf6.ModflowPrtfmi(prt_dr, packagedata=fmi_pd_dr)
        ems_dr = flopy.mf6.ModflowEms(prt_sim_dr, filename=f'{prt_name_dr}.ems')
        prt_sim_dr.register_solution_package(ems_dr, [prt_dr.name])

        prt_sim_dr.write_simulation()
        success_prt_dr, _ = prt_sim_dr.run_simulation()

        if success_prt_dr:
            # Load drought pathlines
            df_dr = pd.read_csv(os.path.join(prt_ws_dr, trackcsv_dr))
            xw_dr, yw_dr = ref_modelgrid.get_coords(df_dr['x'].values, df_dr['y'].values)
            df_dr['x_world'] = xw_dr
            df_dr['y_world'] = yw_dr

            term_dr = df_dr.groupby('irpt').last()
            cap_ids_dr = set(term_dr.loc[term_dr['izone'] == 1].index)
            cap_df_dr = df_dr[df_dr['irpt'].isin(cap_ids_dr)]

            print(f"Drought PRT: {len(cap_ids_dr)} particles captured "
                  f"(vs {len(captured_ids)} under average conditions)")

            # Compute common zoom extent from both capture zones
            all_cap_x = np.concatenate([
                captured_df['x_world'].values, cap_df_dr['x_world'].values
            ])
            all_cap_y = np.concatenate([
                captured_df['y_world'].values, cap_df_dr['y_world'].values
            ])
            pad = 200
            xlim = (min(all_cap_x.min(), well_x) - pad,
                    max(all_cap_x.max(), well_x) + pad)
            ylim = (min(all_cap_y.min(), well_y) - pad,
                    max(all_cap_y.max(), well_y) + pad)

            # Compare capture zones on refined grid
            fig, axes = plt.subplots(1, 2, figsize=(20, 8))
            for ax, cids, cdf, title in zip(
                axes,
                [captured_ids, cap_ids_dr],
                [captured_df, cap_df_dr],
                ['Average Conditions', 'Drought (-30% recharge)'],
            ):
                quick_model_plot(mg_ref, heads_ref_plot, boundary_gdf=boundary_gdf,
                                 cmap='coolwarm', colorbar_label='Head (m)',
                                 title=title, ax=ax)
                for pid in cids:
                    pl = cdf[cdf['irpt'] == pid]
                    ax.plot(pl['x_world'].values, pl['y_world'].values,
                            color='blue', linewidth=0.6, alpha=0.5)
                ax.scatter(well_x, well_y, s=85, c='black', marker='v',
                           edgecolors='white', linewidth=1.5, zorder=6)
                ax.set_xlim(*xlim)
                ax.set_ylim(*ylim)

            fig.suptitle('Capture Zone Comparison: Average vs Drought',
                         fontsize=14, y=1.02)
            fig.tight_layout()
            plt.show()
        else:
            print("Drought PRT simulation failed.")
    else:
        print("GWF drought run failed.")

    # Restore recharge on refined model
    rch_pkg_ref.recharge.set_data(original_rch_ref)
    print("Recharge restored to calibrated values.")
else:
    print("No base pathlines available — skipping drought comparison.")

In [ ]:
# Checkpoint 8: How does the capture zone change under drought?
create_multiple_choice('task08_checkpoint_8')

---

## 5 — Synthesis: Pre-Application Checklist

Before applying the model to any new scenario, verify:

| Item | What we did | |
|------|------------|---|
| **Objective clearly defined** | "What is the drawdown and where does the water come from?" | |
| **Model capable of answering** | Steady-state drawdown and water balance — yes | |
| **Prediction within validated domain** | Western domain is NOT validated (NB6/NB7) — caveat documented | |
| **Scenarios physically plausible** | Q = 500 m³/d is realistic for a municipal well | |
| **Boundary conditions consistent** | River stage and recharge unchanged (except in drought test) | |
| **Uncertainty considered** | NB7 shows high K uncertainty in western domain | |
| **Caveats documented** | Steady-state only, unvalidated area, single K realisation | |
| **Results reproducible** | Fresh workspace copy each run, fixed random seeds | |

**What a professional application would add:**
- Ensemble of K-field realisations (PEST++ IES) for probabilistic capture zones
- Transient simulation for response times
- Sensitivity runs varying K at the well location
- Formal uncertainty quantification on protection zone boundaries

---

## 6 — Summary

**What we did:**
- Applied the **Five Questions** framework to design a pumping well scenario
- Added a well in the data-sparse western domain and computed **steady-state drawdown**
- Showed that the river provides the water (head-dependent boundary response)
- Verified **linearity** of the steady-state flow equation
- Used MODFLOW 6 **PRT** (forward particle tracking) to delineate the well's capture zone
- Compared the numerical capture zone with the **analytical solution** (Bear, 1979)
- Mapped capture zone boundaries onto **Swiss protection zones** (S1, S2, S3)
- *(optional)* Tested **drought impacts** on heads and capture zone extent

**Key findings:**
- Drawdown is modest (~1–3 m) due to the productive alluvial aquifer and river recharge
- The river boundary limits drawdown extent and supplies the pumped water
- The 10-day isochrone (S2) extends ~100–300 m from the well
- Under drought, the capture zone **expands** — protection zones should be based on conservative scenarios

## What You're Taking Forward

| From this notebook | Used in |
|-------------------|---------|
| Scenario design workflow (Five Questions) | Group case study — design your geothermal scenario |
| Drawdown computation and interpretation | Group case study — assess thermal impacts |
| MODFLOW 6 PRT particle tracking | Group case study — thermal breakthrough analysis |
| Protection zone concepts | Professional practice |
| Analytical vs numerical comparison | Validating your own model results |

## Next Steps

**Continue to:** [9_documentation.ipynb](9_documentation.ipynb) — where we learn to document the modelling process for reproducibility and communication.

**Group work:** Apply these same techniques to your geothermal well doublet scenario.

## References

- Anderson, M.P., Woessner, W.W., Hunt, R.J. (2015). *Applied Groundwater Modeling* (2nd ed.). Academic Press.
- Bear, J. (1979). *Hydraulics of Groundwater*. McGraw-Hill. Chapter 8: Wells.
- Pollock, D.W. (2016). User guide for MODPATH Version 7. USGS Open-File Report 2016-1086.
- Swiss Federal Council (2018). *Gewässerschutzverordnung (GSchV)*, SR 814.201. Art. 29: Grundwasserschutzzonen.
- Hill, M.C., Tiedeman, C.R. (2007). *Effective Groundwater Model Calibration*. Wiley.
- Modflow 6 Particle Tracking (PRT): Hughes, J.D., Langevin, C.D., Paulinski, S.R., Larsen, J.D., and Brakenhoff, D. (2023). Particle tracking for MODFLOW 6. USGS Techniques and Methods 6-A78.

In [ ]:
# --- Final cleanup: restore model to baseline state ---
wel.stress_period_data.set_data(existing_wells, key=0)
sim.write_simulation()
print("Model restored to baseline state.")